# Clickbait Headline Detector

## **1. Install dependencies**

In [1]:
%pip install -q torch transformers datasets accelerate evaluate scikit-learn pandas numpy ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 7.3 MB/s eta 0:00:00


## **2. Configuration**

In [2]:
import os

DATASET_NAME = "christinacdl/clickbait_detection_dataset"
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

ID2LABEL = {0: "not_clickbait", 1: "clickbait"}
LABEL2ID = {"not_clickbait": 0, "clickbait": 1}

BASE_MODEL_NAME = "distilbert-base-uncased"
MAX_SEQUENCE_LENGTH = 64

OUTPUT_DIR = os.path.join(os.getcwd(), "clickbait-distilbert")
LOGS_DIR = os.path.join(os.getcwd(), "logs")

SEED = 42
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

print(f"Model will be saved to: {OUTPUT_DIR}")

Model will be saved to: /content/clickbait-distilbert


## **3. Load and split the dataset**

In [3]:
from datasets import load_dataset

raw = load_dataset(DATASET_NAME)

if "validation" in raw and "test" in raw:
    train_ds, val_ds, test_ds = raw["train"], raw["validation"], raw["test"]
else:
    full = raw["train"]
    split_1 = full.train_test_split(test_size=0.2, seed=SEED, stratify_by_column=LABEL_COLUMN)
    split_2 = split_1["test"].train_test_split(test_size=0.5, seed=SEED, stratify_by_column=LABEL_COLUMN)
    train_ds, val_ds, test_ds = split_1["train"], split_2["train"], split_2["test"]

print(f"Train / val / test sizes: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}")
train_ds[0]

README.md:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

train.json:   0%|          | 0.00/2.99M [00:00<?, ?B/s]

val.json:   0%|          | 0.00/375k [00:00<?, ?B/s]

test.json:   0%|          | 0.00/373k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30296 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3787 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3787 [00:00<?, ? examples/s]

Train / val / test sizes: 30296 / 3787 / 3787


{'text_label': 'CLICKBAIT',
 'text': '14 Times People Turned Into Emojis At The Munk Debate',
 'label': 1}

## **4. Tokenize**

In [4]:
# Tokenize all three splits with the DistilBERT tokenizer
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch[TEXT_COLUMN], truncation=True, max_length=MAX_SEQUENCE_LENGTH)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenization complete.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/30296 [00:00<?, ? examples/s]

Map:   0%|          | 0/3787 [00:00<?, ? examples/s]

Map:   0%|          | 0/3787 [00:00<?, ? examples/s]

Tokenization complete.


## **5. Build the model**

In [5]:
# Load DistilBERT with a binary classification head
import torch
from transformers import AutoModelForSequenceClassification, set_seed

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,)

Using device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## **6. Train**

In [6]:
# Metrics
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", zero_division=0
    )
    macro_f1 = f1_score(labels, predictions, average="macro")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "macro_f1": macro_f1,
    }

In [8]:
# Training arguments and Trainer setup
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=LOGS_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1
1,0.079317,0.078883,0.970953,0.976334,0.968531,0.972417,0.970871
2,0.047232,0.087721,0.974650,0.976977,0.975025,0.976000,0.974570
3,0.019055,0.104842,0.974122,0.972691,0.978521,0.975598,0.974027


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2841, training_loss=0.07753925269357506, metrics={'train_runtime': 252.6942, 'train_samples_per_second': 359.676, 'train_steps_per_second': 11.243, 'total_flos': 538873238126400.0, 'train_loss': 0.07753925269357506, 'epoch': 3.0})

## **7. Evaluate on the test set**

In [10]:
test_metrics = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test")
for key, value in test_metrics.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

[transformers] early stopping required metric_for_best_model, but did not find eval_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Macro F1
0.019055,0.088992,3,0.973594,0.973606,0.976523,0.975062,0.973502


test_loss: 0.0890
test_accuracy: 0.9736
test_precision: 0.9736
test_recall: 0.9765
test_f1: 0.9751
test_macro_f1: 0.9735


## **8. Save the model**

In [11]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model and tokenizer saved to: {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to: /content/clickbait-distilbert


## **9. Run predictions**

In [12]:
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification as _AutoModel
from transformers import AutoTokenizer as _AutoTokenizer

_inference_device = "cuda" if torch.cuda.is_available() else "cpu"
_inference_tokenizer = _AutoTokenizer.from_pretrained(OUTPUT_DIR)
_inference_model = _AutoModel.from_pretrained(OUTPUT_DIR).to(_inference_device)
_inference_model.eval()

@torch.no_grad()
def predict(text):
    inputs = _inference_tokenizer(
        text, truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors="pt"
    ).to(_inference_device)

    logits = _inference_model(**inputs).logits
    probs = F.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_id = int(probs.argmax())

    return {
        "text": text,
        "label": ID2LABEL[pred_id],
        "confidence": float(probs[pred_id]),
        "probabilities": {ID2LABEL[i]: float(p) for i, p in enumerate(probs)},
    }

predict("You Won't Believe What This DistilBERT Model Just Did")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'text': "You Won't Believe What This DistilBERT Model Just Did",
 'label': 'clickbait',
 'confidence': 0.9996170997619629,
 'probabilities': {'not_clickbait': 0.0003828447952400893,
  'clickbait': 0.9996170997619629}}

## **10. UI**



In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output

headline_input = widgets.Textarea(
    placeholder="Type or paste a headline here",
    layout=widgets.Layout(width="100%", height="80px"),
)
check_button = widgets.Button(description="Check headline", button_style="primary")
output_area = widgets.Output()

def on_check_clicked(_):
    with output_area:
        clear_output()
        text = headline_input.value.strip()
        if not text:
            print("Enter a headline first.")
            return

        result = predict(text)
        verdict = "CLICKBAIT" if result["label"] == "clickbait" else "GENUINE"
        print(f"Verdict:    {verdict}")
        print(f"Confidence: {result['confidence']:.1%}")
        print("Probabilities:")
        for label, prob in result["probabilities"].items():
            bar = "#" * int(prob * 30)
            print(f"  {label:>14}: {prob:.1%}  {bar}")

check_button.on_click(on_check_clicked)

display(widgets.VBox([headline_input, check_button, output_area]))